In [1]:
!pip install transformers

In [2]:
# Importing stock ml libraries
import warnings
warnings.simplefilter('ignore')
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn import metrics
import transformers
import torch
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from transformers import DistilBertTokenizer, DistilBertModel
import logging
logging.basicConfig(level=logging.ERROR)

In [3]:
fos_df = pd.read_excel('/content/Train_Set.xlsx')
fos_df.head()

,Usage,Is FOS
0,Ang iyang kauban sa trabaho pirmi magpalapad o...,1
1,Nagpalit siya og balayronon sa papel sa tindah...,0
2,"""Ang silingan murag namiya og tai, kalit lang ...",1
3,Ang bata nagdali sa pagdagan kay murag namiya ...,0
4,"Si Pedro makawat kaayo, dali ra siya madala bi...",1


In [4]:
fos_df.columns.to_list()

['Usage', 'Is FOS']

In [5]:
fos_df = fos_df.rename(columns={"Is FOS": "is_fos"})

In [6]:
fos_df.head()

,Usage,is_fos
0,Ang iyang kauban sa trabaho pirmi magpalapad o...,1
1,Nagpalit siya og balayronon sa papel sa tindah...,0
2,"""Ang silingan murag namiya og tai, kalit lang ...",1
3,Ang bata nagdali sa pagdagan kay murag namiya ...,0
4,"Si Pedro makawat kaayo, dali ra siya madala bi...",1


In [7]:
from transformers import pipeline

#unmasker = pipeline('fill-mask', model='dost-asti/BERT-ceb-cased')
unmasker = pipeline('fill-mask', model='GianTan/CBERTo')
unmasker("Hello I'm a [MASK] model.")

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/231k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


[{'score': 0.017744773998856544,
  'token': 457,
  'token_str': 'sentence',
  'sequence': "hello i'm a sentence model."},
 {'score': 0.013290436938405037,
  'token': 1461,
  'token_str': 'given',
  'sequence': "hello i'm a given model."},
 {'score': 0.010848677717149258,
  'token': 4051,
  'token_str': 'story',
  'sequence': "hello i'm a story model."},
 {'score': 0.009323697537183762,
  'token': 3482,
  'token_str': 'text',
  'sequence': "hello i'm a text model."},
 {'score': 0.00910557247698307,
  'token': 1241,
  'token_str': 'list',
  'sequence': "hello i'm a list model."}]

In [8]:
tokenizer = DistilBertTokenizer.from_pretrained('GianTan/CBERTo',truncation=True, do_lower_case=False)
model = DistilBertModel.from_pretrained("GianTan/CBERTo")

In [27]:
input = tokenizer('Murag mag bata nis Gian.', return_tensors='pt')
input

{'input_ids': tensor([[   2,    1,  223,  770, 4073,    1,   20,    3]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}

In [10]:
output = model(input_ids=input['input_ids'], attention_mask=input['attention_mask'])
output

BaseModelOutput(last_hidden_state=tensor([[[ 2.1285, -2.3344, -0.3507,  ..., -0.3936,  0.6141, -0.0684],
         [ 0.4428, -0.5441,  0.1129,  ...,  1.1983,  0.8404,  0.9629],
         [-0.9604,  0.2518,  0.1822,  ..., -0.6666,  1.2541, -2.0116],
         ...,
         [-1.0601, -1.7673, -0.3557,  ...,  0.3688,  1.0913, -0.8512],
         [-1.4599, -0.7658, -2.2719,  ..., -0.5873,  0.6600, -1.8172],
         [-1.3879, -0.4668, -0.8622,  ...,  0.2827, -1.2396,  0.1699]]],
       grad_fn=<NativeLayerNormBackward0>), hidden_states=None, attentions=None)

In [11]:
# Sections of config

# Defining some key variables that will be used later on in the training
MAX_LEN = 64
TRAIN_BATCH_SIZE = 128
VALID_BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 4e-05

In [12]:
class FOS_Classification(Dataset):

    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.text = dataframe.Usage
        self.targets = self.data.is_fos
        self.max_len = max_len

    def __len__(self):
        return len(self.text)

    def __getitem__(self, index):
        text = str(self.text[index])
        text = " ".join(text.split())

        inputs = self.tokenizer.encode_plus(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            pad_to_max_length=True,
            return_token_type_ids=True
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']
        token_type_ids = inputs["token_type_ids"]


        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
            'targets': torch.tensor(self.targets[index], dtype=torch.float)
        }

In [13]:
# Creating the dataset and dataloader for the neural network

train_size = 0.8
train_data=fos_df.sample(frac=train_size,random_state=69)
test_data=fos_df.drop(train_data.index).reset_index(drop=True)
train_data = train_data.reset_index(drop=True)


print("FULL Dataset: {}".format(fos_df.shape))
print("TRAIN Dataset: {}".format(train_data.shape))
print("TEST Dataset: {}".format(test_data.shape))

training_set = FOS_Classification(train_data, tokenizer, MAX_LEN)
testing_set = FOS_Classification(test_data, tokenizer, MAX_LEN)

FULL Dataset: (1288, 2)
TRAIN Dataset: (1159, 2)
TEST Dataset: (129, 2)


In [14]:
train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

training_loader = DataLoader(training_set, **train_params)
testing_loader = DataLoader(testing_set, **test_params)

In [15]:
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'

In [16]:
# Creating the customized model, by adding a drop out and a dense layer on top of distil bert to get the final output for the model.

class BertClass(torch.nn.Module):
    def __init__(self):
        super(BertClass, self).__init__()
        self.l1 = DistilBertModel.from_pretrained('GianTan/CBERTo')
        self.pre_classifier = torch.nn.Linear(768, 768)
        self.dropout = torch.nn.Dropout(0.3)

        self.pre_classifier2 = torch.nn.Linear(768, 768)
        self.dropout2 = torch.nn.Dropout(0.3)



        self.classifier = torch.nn.Linear(768,1)

    def forward(self, input_ids, attention_mask, token_type_ids):
        output_1 = self.l1(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = output_1[0]
        pooler = hidden_state[:, 0]
        pooler = self.pre_classifier(pooler)
        pooler = torch.nn.Tanh()(pooler)
        pooler = self.dropout(pooler)
        pooler = self.pre_classifier2(pooler)
        pooler = torch.nn.Tanh()(pooler)
        pooler = self.dropout2(pooler)


        output = self.classifier(pooler)
        return output.squeeze(1)



model = BertClass()
model.to(device)

BertClass(
  (l1): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30000, 768, padding_idx=0)
      (position_embeddings): Embedding(256, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(in_feat

In [17]:
def loss_fn(outputs, targets):
    return torch.nn.BCEWithLogitsLoss()(outputs, targets)

In [18]:
optimizer = torch.optim.Adam(params =  model.parameters(), lr=LEARNING_RATE)

In [19]:
def train(epoch):
    model.train()
    for _,data in tqdm(enumerate(training_loader, 0)):
        ids = data['ids'].to(device, dtype = torch.long)
        mask = data['mask'].to(device, dtype = torch.long)
        token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
        targets = data['targets'].to(device, dtype = torch.float)

        outputs = model(ids, mask, token_type_ids)

        optimizer.zero_grad()
        loss = loss_fn(outputs, targets)
        if _%5000==0:
            print(f'Epoch: {epoch}, Loss:  {loss.item()}')

        loss.backward()
        optimizer.step()

In [20]:
for epoch in range(EPOCHS):
    train(epoch)

0it [00:00, ?it/s]Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Epoch: 0, Loss:  0.6863101720809937


10it [00:05,  1.73it/s]
1it [00:00,  1.92it/s]

Epoch: 1, Loss:  0.5570710897445679


10it [00:05,  1.93it/s]
1it [00:00,  3.30it/s]

...

10it [00:05,  1.96it/s]
1it [00:00,  4.08it/s]

Epoch: 8, Loss:  0.03512043505907059


10it [00:05,  1.92it/s]
1it [00:00,  3.08it/s]

Epoch: 9, Loss:  0.026060540229082108


10it [00:05,  1.89it/s]


In [21]:

def test_model(item):
  input_text = item
  encoded_text = tokenizer.encode_plus(
      input_text,
      None,
      add_special_tokens=True,
      max_length=MAX_LEN,
      pad_to_max_length=True,
      return_token_type_ids=True
  )

  # Convert the input to tensors
  input_ids = torch.tensor(encoded_text['input_ids']).unsqueeze(0)
  input_mask = torch.tensor(encoded_text['attention_mask']).unsqueeze(0)
  segment_ids = torch.tensor(encoded_text['token_type_ids']).unsqueeze(0)

  # Move tensors to the device
  input_ids = input_ids.to(device)
  input_mask = input_mask.to(device)
  segment_ids = segment_ids.to(device)

  # Make predictions
  with torch.no_grad():
      outputs = model(input_ids, input_mask, segment_ids)

  # Apply sigmoid activation function
  outputs = torch.sigmoid(outputs)

  # Convert the outputs to numpy array
  outputs = outputs.cpu().detach().numpy()

  return outputs

In [22]:
fos_df

,Usage,is_fos
0,Ang iyang kauban sa trabaho pirmi magpalapad o...,1
1,Nagpalit siya og balayronon sa papel sa tindah...,0
2,"""Ang silingan murag namiya og tai, kalit lang ...",1
3,Ang bata nagdali sa pagdagan kay murag namiya ...,0
4,"Si Pedro makawat kaayo, dali ra siya madala bi...",1
...,...,...
1283,Ang bata nalipay nga gidawat limpyo ang bag-on...,0
1284,Naa sila sa tunga sa ilang paglakaw padulong s...,1
1285,"""Pag-abot nako sa balay, akong gibutangan ug t...",0
1286,Si Maria nagstruggle kay pirmi malimtan ang mg...,1


In [23]:
test_model("Ang bata nalipay nga gidawat limpyo ")

array([0.00534666], dtype=float32)

In [24]:
#  from the validation function, let me see the actual string being passed and not just the number or tensor conversion

def validation(testing_loader):
    model.eval()
    fin_targets=[]
    fin_outputs=[]

    with torch.no_grad():
        for _, data in tqdm(enumerate(testing_loader, 0)):
            ids = data['ids'].to(device, dtype = torch.long)
            mask = data['mask'].to(device, dtype = torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
            targets = data['targets'].to(device, dtype = torch.float)
            outputs = model(ids, mask, token_type_ids)
            fin_targets.extend(targets.cpu().detach().numpy().tolist())
            fin_outputs.extend(torch.sigmoid(outputs).cpu().detach().numpy().tolist())

    return fin_outputs, fin_targets


In [25]:
outputs, targets = validation(testing_loader)

final_outputs = np.array(outputs) >= 0.9

print(final_outputs)
print(targets)

2it [00:00,  6.18it/s]

[ True  True False  True False False  True False  True False  True  True
 False False False  True  True False  True  True False  True False  True
  True False  True False False  True False  True  True  True  True False
  True False False False False  True  True False False False  True False
  True False False False False  True  True False False False False False
  True  True False False  True False False  True False  True  True  True
  True False  True False  True False False  True  True False False  True
  True False False False  True False  True False False  True  True  True
  True False False  True  True  True False False False  True False  True
  True False False  True False False False  True False False False  True
  True False False False False  True  True False False]
[1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0

In [26]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

score_dat = f1_score(np.array(targets),np.array(final_outputs),average='weighted')
print(score_dat)

y_pred = np.array(targets)
y_true = np.array(final_outputs)

print(accuracy_score(y_true, y_pred))

0.8994821827986641
0.8992248062015504
